# DeepTaxa – Tutorial 2: Training
This notebook reproduces the [official training tutorial](https://systems-genomics-lab.github.io/deeptaxa/training.html).

It covers data preparation, hyperparameter configuration, launching a training run,
and evaluating the trained model.

In [ ]:
!pip install -q deeptaxa huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs/checkpoints', exist_ok=True)

In [ ]:
# Download Greengenes2 training data from HuggingFace
from huggingface_hub import hf_hub_download
train_path = hf_hub_download(
    repo_id='systems-genomics-lab/deeptaxa',
    filename='data/gg2_train.fna',
    local_dir='data'
)
test_path = hf_hub_download(
    repo_id='systems-genomics-lab/deeptaxa',
    filename='data/gg2_test.fna',
    local_dir='data'
)
print('Train:', train_path)
print('Test: ', test_path)

In [ ]:
# Count sequences
def count_seqs(path):
    return sum(1 for l in open(path) if l.startswith('>'))
print(f'Train sequences: {count_seqs(train_path):,}')
print(f'Test  sequences: {count_seqs(test_path):,}')

In [ ]:
# Visualise the cosine-annealing learning-rate schedule
import numpy as np, matplotlib.pyplot as plt
epochs = 30
lr_max, lr_min = 1e-4, 1e-6
lrs = [lr_min + 0.5*(lr_max-lr_min)*(1+np.cos(np.pi*e/epochs)) for e in range(epochs)]
plt.figure(figsize=(8,4))
plt.plot(range(1, epochs+1), lrs, marker='o', ms=4)
plt.xlabel('Epoch'); plt.ylabel('Learning rate')
plt.title('Cosine-annealing LR schedule (DeepTaxa default)')
plt.tight_layout(); plt.show()

In [ ]:
# Training hyperparameters (DeepTaxa defaults)
import pandas as pd
params = {
    'batch_size': 32,
    'learning_rate': 1e-4,
    'num_epochs': 30,
    'max_length': 512,
    'embed_dim': 896,
    'hidden_size': 896,
    'num_hidden_layers': 4,
    'num_attention_heads': 7,
    'intermediate_size': 3584,
    'num_filters': 256,
    'kernel_sizes': '[3,5,7]',
    'num_conv_layers': 3,
    'hidden_dropout_prob': 0.1,
    'weight_decay': 0.01,
    'warmup_steps': 500,
}
pd.DataFrame(params.items(), columns=['Parameter','Value'])

In [ ]:
# Launch training (GPU recommended – set batch-size lower on free Colab T4)
!deeptaxa train \
    --train-data data/gg2_train.fna \
    --test-data  data/gg2_test.fna \
    --output-dir outputs/checkpoints \
    --epochs 5 \
    --batch-size 16 \
    --learning-rate 1e-4 \
    --tokenizer zhihan1996/DNABERT-2-117M

In [ ]:
# Plot training and validation loss curves
import json, glob, matplotlib.pyplot as plt
log_files = glob.glob('outputs/checkpoints/training_log*.json')
if log_files:
    with open(log_files[-1]) as f: log = json.load(f)
    fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12,4))
    ax1.plot(log.get('train_loss',[]), label='train')
    ax1.plot(log.get('val_loss',[]),   label='val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
    ax1.set_title('Loss curves')
    ax2.plot(log.get('train_f1',[]), label='train')
    ax2.plot(log.get('val_f1',[]),   label='val')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro F1'); ax2.legend()
    ax2.set_title('Macro F1 curves')
    plt.tight_layout(); plt.show()
else:
    print('No log file found – run training first.')

In [ ]:
# Evaluate the trained checkpoint
import glob
ckpts = glob.glob('outputs/checkpoints/*.pt')
if ckpts:
    best = sorted(ckpts)[-1]
    !deeptaxa evaluate \
        --model-path {best} \
        --test-data  data/gg2_test.fna \
        --batch-size 32
else:
    print('No checkpoint found – run training first.')